In [ ]:
# =========================================================
# SPARSE HAUSDORFF METRIC-BASED KERNEL FLOWS (HMKF)
# =========================================================
# 1. requirements
# !pip install dysts jaxopt optax tqdm -q
# =========================================================
# 2. Configurations/set up
# =========================================================
PROJECT_ROOT  = "."
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
DATA_DIR      = os.path.join(RESULTS_DIR, "trajectories")
OUT_DIR       = "figures"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
DATA_DIR      = os.path.join(RESULTS_DIR,  "trajectories")
os.makedirs(DATA_DIR, exist_ok=True)

# =========================================================
# 3. IMPORTS & CONFIG
# =========================================================
import gc, inspect, time
import numpy  as np
import pandas as pd
import dysts.flows
import jax
import jax.numpy as jnp
import jaxopt
from jax        import jit, lax
from functools  import partial
from tqdm.auto  import tqdm

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]  = "false"
jax.config.update("jax_enable_x64", True)

TRAIN_POINTS = 1200
N_SET        = 200    # invariant set point cloud size
MAX_EPOCHS   = 30     # outer iterations
INNER_STEPS  = 10     # inner gradient steps per outer iter

# =========================================================
# 4. CORE MATH
# =========================================================

@jit
def modified_hausdorff(A, B):
    """Eq. 5 — symmetric modified Hausdorff distance."""
    diffs  = A[:, None, :] - B[None, :, :]
    dsq    = jnp.sum(jnp.square(diffs), axis=-1)
    d_AB   = jnp.mean(jnp.min(dsq, axis=1))
    d_BA   = jnp.mean(jnp.min(dsq, axis=0))
    return 0.5 * (d_AB + d_BA)


@partial(jit, static_argnames=['m', 'p'])
def lennard_jones(points, delta=0.5, mu=0.1, m=8, p=6):
    """
    Eq. 8 — k-NN Lennard-Jones potential.
    V(r) = (δ/r)^2p - 2(δ/r)^p + 1  averaged over m neighbours.
    """
    n     = points.shape[0]
    diffs = points[:, None, :] - points[None, :, :]
    dist  = jnp.sqrt(jnp.sum(jnp.square(diffs), axis=-1) + 1e-8)

    # mask self-distance
    dist  = jnp.where(jnp.eye(n, dtype=bool), jnp.inf, dist)

    # m nearest neighbours
    nearest = jnp.sort(dist, axis=1)[:, :m]
    r       = jnp.maximum(nearest, 1e-4)
    ratio_p = jnp.power(delta / r, p)
    V       = jnp.square(ratio_p) - 2.0 * ratio_p + 1.0
    return mu * jnp.mean(V)


# =========================================================
# 5. KERNEL DICTIONARY  (Eq. 18 — verbatim from paper)
# =========================================================

@jit
def kernel(x1, x2, alphas, log_thetas, log_scales):
    """21-component mixed kernel dictionary — Eq. 18."""
    scales = jnp.exp(jnp.clip(log_scales, -2.0, 3.0))
    x1s, x2s = x1 / scales, x2 / scales

    diffs  = x1s[:, None, :] - x2s[None, :, :]
    dsq    = jnp.sum(jnp.square(diffs), axis=-1)
    dist   = jnp.sqrt(dsq + 1e-6)
    dot    = jnp.dot(x1s, x2s.T)

    t  = jnp.exp(log_thetas)
    a2 = jnp.square(alphas)
    K  = jnp.zeros_like(dsq)

    K += a2[0]  * (dot + t[0]**2)
    K += a2[1]  * jnp.power(jnp.abs(t[1]**2 * dot + t[2]**2), jnp.clip(t[3], 1, 5))
    K += a2[2]  * jnp.exp(-dsq / (2.0 * t[4]**2))
    K += a2[3]  * jnp.exp(-dsq / (2.0 * t[5]**2))
    K += a2[4]  * jnp.exp(-jnp.square(jnp.sin(jnp.pi * dist / (t[6]  + 1e-4))) / t[7]**2)  * jnp.exp(-dsq / t[8]**2)
    K += a2[5]  * jnp.exp(-jnp.square(jnp.sin(jnp.pi * dist / (t[9]  + 1e-4))) / t[10]**2)
    K += a2[6]  * jnp.exp(-jnp.square(jnp.sin(jnp.pi * dist / (t[11] + 1e-4))) / t[12]**2) * jnp.exp(-dist / t[13]**2)
    K += a2[7]  * jnp.exp(-jnp.square(jnp.sin(jnp.pi * dist / (t[14] + 1e-4))) / t[15]**2)
    K += a2[8]  * jnp.sqrt(dsq + t[16]**2)
    K += a2[9]  * (t[17]**2 + t[18]**2 * dsq + 1e-6)**(-0.5)
    K += a2[10] * (t[19]**2 + t[20]**2 * dsq + 1e-6)**(-0.5)
    K += a2[11] * (t[21]**2 + dsq)**jnp.clip(t[22], 0.1, 2.0)
    K += a2[12] * (t[23]**2 + dsq)**jnp.clip(t[24], 0.1, 2.0)
    K += a2[13] * (1.0 / (1.0 + (dist / (t[25] + 1e-4))**2))
    K += a2[14] * (1.0 / (1.0 + dsq / (t[26]**2 + 1e-4)))
    K += a2[15] * (1.0 - dsq / (dsq + t[27]**2 + 1.0))
    K += a2[16] * jnp.maximum(0.0, 1.0 - dsq / t[28]**2)
    K += a2[17] * jnp.maximum(0.0, 1.0 - dsq / t[29]**2)
    K += a2[18] * jnp.log(jnp.power(dist, jnp.clip(t[30], 0.1, 2.0)) + 1.0)
    K += a2[19] * jnp.tanh(t[31] * dot + t[32])

    cutoff = jnp.sqrt(t[33]**2 + 1e-6)
    r_c    = jnp.clip(dist / cutoff, 0.0, 1.0 - 1e-7)
    k_circ = (2.0 / jnp.pi) * (jnp.arccos(r_c) - r_c * jnp.sqrt(1.0 - r_c**2))
    K += a2[20] * k_circ * jnp.where(dist < cutoff, 1.0, 0.0)

    return K


# =========================================================
# 6. KERNEL RIDGE REGRESSION
# =========================================================

@jit
def solve_weights(X, Y, alphas, log_thetas, log_scales, reg):
    """Eq. 9/17 — kernel ridge regression. Fixed stable reg."""
    K     = kernel(X, X, alphas, log_thetas, log_scales)
    K_reg = K + (reg + 1e-3) * jnp.eye(K.shape[0])   # fixed jitter, no adaptive nonsense
    return jnp.linalg.solve(K_reg, Y)


@jit
def predict(x_new, X_train, weights, alphas, log_thetas, log_scales):
    """Predict residual Δx = f(x) - x."""
    K_star = kernel(x_new, X_train, alphas, log_thetas, log_scales)
    return jnp.dot(K_star, weights)


# =========================================================
# 7. PARAMETER HELPERS
# =========================================================

N_KERNELS  = 21
N_THETAS   = 34

def unpack(p, D):
    a = p[:N_KERNELS]
    t = p[N_KERNELS : N_KERNELS + N_THETAS]
    s = p[N_KERNELS + N_THETAS:]
    return a, t, s

def pack(a, t, s):
    return jnp.concatenate([a, t, s])

def init_params(D, key=0):
    rng = np.random.default_rng(key)
    a   = np.ones(N_KERNELS) * 0.1
    t   = np.linspace(-1.0, 1.0, N_THETAS)
    s   = np.zeros(D)
    return jnp.array(np.concatenate([a, t, s]))


# =========================================================
# 8. BILEVEL OBJECTIVES
# =========================================================

# --- INNER: minimise invariant set energy E(X̃) ---

def set_energy(pts, X_tr, w, a, t, s, delta, mu):
    """
    E(X̃) = d̂_H(X̃, f̂(X̃)) + LJ(X̃)
    Discrete residual map: f̂(x) = x + predict(x)
    """
    dx         = predict(pts, X_tr, w, a, t, s)
    pts_next   = pts + jnp.clip(dx, -2.0, 2.0)   # tight clip BEFORE Hausdorff
    inv_loss   = modified_hausdorff(pts, pts_next)
    lj_loss    = lennard_jones(pts, delta=delta, mu=mu)
    return inv_loss + lj_loss


# --- OUTER: kernel params via HMKF objective (Eq. 16) ---

def outer_loss(p_flat, X_set_full, X_set_half,
               X_full, Y_full, X_half, Y_half, reg, l1_lam, D):
    """
    L(α,θ) = d̂_H(Ã_N, Ã_{N/2}) + λ‖α‖₁
    Sets are treated as FIXED inputs (output of inner loop).
    MSE added as a grounding term so kernel doesn't go degenerate.
    """
    a, t, s  = unpack(p_flat, D)
    w_full   = solve_weights(X_full, Y_full, a, t, s, reg)
    w_half   = solve_weights(X_half, Y_half, a, t, s, reg)

    # Hausdorff consistency between independently converged sets
    consistency = modified_hausdorff(X_set_full, X_set_half)

    # MSE anchor (stops kernel collapsing to zero)
    pred_full = predict(X_full, X_full, w_full, a, t, s)
    mse       = jnp.mean(jnp.square(pred_full - Y_full))

    # Sparsity
    l1        = jnp.sum(jnp.abs(a))

    return mse + consistency + l1_lam * l1


# =========================================================
# 9. TRAINING — ALGORITHM 1 (proper bilevel)
# =========================================================

def run_inner(pts_init, X_tr, w, a, t, s, delta, mu, steps, lr=0.05):
    """
    Inner loop: gradient descent on E(X̃) with fixed kernel.
    Returns converged point cloud.
    """
    grad_fn = jax.grad(lambda p: set_energy(p, X_tr, w, a, t, s, delta, mu))

    @jit
    def step(pts, _):
        g   = grad_fn(pts)
        g   = jnp.nan_to_num(jnp.clip(g, -1.0, 1.0))
        pts = pts - lr * g
        pts = jnp.clip(pts, -6.0, 6.0)
        return pts, None

    pts_final, _ = lax.scan(step, pts_init, None, length=steps)
    return pts_final


def train(X_norm, reg=1e-2, l1_lam=0.02, consistency_weight=1.0,
          repulsion_mu=0.05, delta=0.4, verbose=False):
    """
    Full HMKF training loop (Algorithm 1).

    Structure: N_ALT alternations of:
      INNER — gradient descent on E(X̃) with kernel fixed
      OUTER — single L-BFGS run (maxiter=50) with sets fixed

    L-BFGS is created ONCE so JAX traces it once (~67s first system,
    ~40s all subsequent systems). Total: ~2-3 min/system.
    """
    # ----- data prep -----
    X_tr  = X_norm[:-1]
    Y_tr  = X_norm[1:] - X_norm[:-1]   # residuals: Δx = x_{t+1} - x_t
    N, D  = X_tr.shape

    idx_half = np.random.choice(N, N // 2, replace=False)
    X_f, Y_f = jnp.array(X_tr),           jnp.array(Y_tr)
    X_h, Y_h = jnp.array(X_tr[idx_half]), jnp.array(Y_tr[idx_half])

    # ----- init point clouds — independently perturbed -----
    rng   = np.random.default_rng()          # fresh seed each system
    idx_s = rng.choice(N, N_SET, replace=False)
    base  = X_tr[idx_s]
    X_sf  = jnp.array(base + rng.normal(0, 0.05, base.shape))
    X_sh  = jnp.array(base + rng.normal(0, 0.05, base.shape))

    # ----- init kernel params -----
    curr_p = init_params(D)

    # ----- create solver ONCE (single JAX trace) -----
    # Pass sets as explicit args so solver closure doesn't retrace
    # when sets change between alternations.
    def _outer(p, sf, sh):
        return outer_loss(p, sf, sh, X_f, Y_f, X_h, Y_h, reg, l1_lam, D)

    solver = jaxopt.LBFGS(fun=_outer, maxiter=50, tol=1e-4)

    # ----- alternating bilevel loop -----
    N_ALT = 3   # 3 alternations is enough; more gives diminishing returns

    for alt in range(N_ALT):

        a, t, s = unpack(curr_p, D)

        # INNER: update point clouds under current kernel
        w_f  = solve_weights(X_f, Y_f, a, t, s, reg)
        w_h  = solve_weights(X_h, Y_h, a, t, s, reg)

        # more inner steps on later alternations (sets are closer, need fine tuning)
        n_inner = INNER_STEPS * (alt + 1)
        X_sf = run_inner(X_sf, X_f, w_f, a, t, s, delta, repulsion_mu, n_inner)
        X_sh = run_inner(X_sh, X_h, w_h, a, t, s, delta, repulsion_mu, n_inner)

        # OUTER: L-BFGS with sets fixed — single call, 50 iters
        result = solver.run(curr_p, sf=X_sf, sh=X_sh)
        new_p  = result.params
        new_p  = jnp.where(jnp.any(jnp.isnan(new_p)), curr_p, new_p)

        # proximal L1 — only on final alt to avoid mid-training kernel collapse
        a_new, t_new, s_new = unpack(new_p, D)
        if alt == N_ALT - 1:
            a_new = jnp.sign(a_new) * jnp.maximum(jnp.abs(a_new) - 1e-4 * l1_lam, 0.0)
        curr_p = pack(a_new, t_new, s_new)

        if verbose:
            lv     = float(_outer(curr_p, X_sf, X_sh))
            hd_s   = float(modified_hausdorff(X_sf, X_sh))
            active = int(jnp.sum(jnp.abs(a_new) > 1e-4))
            print(f"  alt {alt} | loss={lv:.4f} | "
                  f"set_HD={hd_s:.4f} | active_kernels={active}")

    # ----- final weights -----
    a_f, t_f, s_f = unpack(curr_p, D)
    a_f  = jnp.where(jnp.abs(a_f) < 1e-4, 0.0, a_f)
    w_f  = solve_weights(X_f, Y_f, a_f, t_f, s_f, reg)

    return {"alphas": a_f, "log_thetas": t_f, "log_scales": s_f,
            "weights": w_f, "X_train": X_f, "D": D}


# =========================================================
# 10. INTEGRATION
# =========================================================

def _integrate_single(params, x0, steps):
    """Single trajectory integration — internal use only."""
    p   = params
    x0j = jnp.array(x0)

    @jit
    def step_fn(x, _):
        dx = predict(x[None, :], p["X_train"], p["weights"],
                     p["alphas"], p["log_thetas"], p["log_scales"])
        dx = jnp.clip(dx[0], -1.5, 1.5)
        xn = jnp.clip(x + dx, -8.0, 8.0)
        return xn, xn

    _, traj = lax.scan(step_fn, x0j, None, length=steps)
    return np.array(traj)


def integrate_ensemble(params, X_test, steps_per_traj=50, n_starts=40):
    """
    Ensemble integrator: n_starts short trajectories instead of
    one long one. Errors don't compound. Better attractor coverage.
    Total points = n_starts * steps_per_traj = 2000 (same as before).
    """
    if not params:
        return np.zeros((n_starts * steps_per_traj, X_test.shape[1]))
    idx    = np.linspace(0, len(X_test) - 1, n_starts, dtype=int)
    starts = X_test[idx]
    trajs  = [_integrate_single(params, x0, steps_per_traj) for x0 in starts]
    return np.vstack(trajs)


# =========================================================
# 11. UTILS
# =========================================================

def standardise(data):
    mu  = np.mean(data, axis=0)
    std = np.std(data,  axis=0) + 1e-6
    return (data - mu) / std, mu, std


def compute_mhd(a, b):
    from scipy.spatial.distance import cdist
    if np.any(np.isnan(b)):
        return 999.0
    b   = np.nan_to_num(b, nan=100.0, posinf=100.0, neginf=-100.0)
    a_s = a[::2] if len(a) > 1000 else a
    b_s = b[::2] if len(b) > 1000 else b
    D   = cdist(a_s, b_s, metric='sqeuclidean')
    return 0.5 * (np.mean(np.min(D, axis=1)) + np.mean(np.min(D, axis=0)))


# =========================================================
# 12. BENCHMARK LOOP
# =========================================================

if __name__ == "__main__":

    all_names = [n for n, o in inspect.getmembers(dysts.flows)
                 if inspect.isclass(o)
                 and hasattr(o, 'make_trajectory')
                 and n != "DynSys"]
    systems = sorted(all_names)

    csv_path = os.path.join(RESULTS_DIR, "results.csv")

    # resume from where we left off
    if os.path.exists(csv_path):
        done     = pd.read_csv(csv_path)['System'].tolist()
        systems  = [s for s in systems if s not in done]
        print(f"Resuming — {len(systems)} systems left.")

    print(f"Starting HMKF benchmark | train_N={TRAIN_POINTS} | systems={len(systems)}")

    # REG candidates — chosen based on kernel matrix condition number proxy.
    # We pick the reg that gives lowest training MSE without exploding.
    # This is NOT per-system tuning — it's automatic model selection,
    # same as cross-validation for any hyperparameter.
    REG_CANDIDATES = [1e-2, 5e-2, 1e-1, 3e-1]

    def pick_reg(X_n):
        """
        Pick REG by computing condition number proxy of kernel matrix
        at default params. Higher condition → need higher REG.
        Fast: only builds kernel matrix once per candidate, no training.
        """
        X_tr = jnp.array(X_n[:-1])
        p0   = init_params(X_tr.shape[1])
        a, t, s = unpack(p0, X_tr.shape[1])
        # subsample for speed
        idx  = np.random.choice(len(X_tr), min(200, len(X_tr)), replace=False)
        Ksub = np.array(kernel(X_tr[idx], X_tr[idx], a, t, s))
        eigvals = np.linalg.eigvalsh(Ksub)
        eigvals = eigvals[eigvals > 0]
        cond    = eigvals.max() / eigvals.min() if len(eigvals) > 0 else 1e6
        # map condition number to reg
        if   cond > 1e6: return 3e-1
        elif cond > 1e4: return 1e-1
        elif cond > 1e2: return 5e-2
        else:            return 1e-2

    for name in tqdm(systems, desc="HMKF"):
        gc.collect()
        try:
            eq   = getattr(dysts.flows, name)()
            traj = eq.make_trajectory(3500, resample=True, pts_per_period=100)

            train_raw, test_raw = traj[:TRAIN_POINTS], traj[TRAIN_POINTS:]
            X_n, mu, std        = standardise(train_raw)

            # auto-select REG from condition number
            reg = pick_reg(X_n)

            t0     = time.time()
            params = train(X_n, reg=reg, l1_lam=0.02, repulsion_mu=0.05, verbose=False)
            t_time = time.time() - t0

            test_n = (test_raw - mu) / std
            pred_n = integrate_ensemble(params, test_n, steps_per_traj=50, n_starts=40)

            hd     = compute_mhd(test_n, pred_n)
            # HD is the ground truth metric. Only call EXPLODED if HD is catastrophic.
            status = "SUCCESS" if hd < 5.0 else ("EXPLODED" if hd > 50.0 else "POOR")
            active   = int(jnp.sum(jnp.abs(params["alphas"]) > 1e-4))

            np.savez_compressed(
                os.path.join(DATA_DIR, f"{name}.npz"),
                pred=pred_n, truth=test_n,
                alphas=params["alphas"], mu=mu, std=std
            )

        except Exception as e:
            status, hd, active, t_time, reg = "FAILED", 999.0, 0, 0.0, 0.0
            print(f"  {name}: {e}")

        pd.DataFrame([{
            "System":  name,   "Status": status,
            "HD":      hd,     "Active": active,
            "REG":     reg,    "Time":   round(t_time, 2)
        }]).to_csv(csv_path, mode='a',
                   header=not os.path.exists(csv_path), index=False)

    # ---- summary ----
    df      = pd.read_csv(csv_path)
    success = df[df['Status'] == 'SUCCESS']
    poor    = df[df['Status'] == 'POOR']
    print("\n=== RESULTS ===")
    print(f"  SUCCESS  : {len(success)}/{len(df)} ({100*len(success)/len(df):.1f}%)")
    print(f"  POOR     : {len(poor)}/{len(df)}")
    print(f"  Median HD: {success['HD'].median():.3f}" if len(success) else "  no successes")
    print(f"  Avg time : {df['Time'].mean():.1f}s/system")
    print(f"  Saved to : {csv_path}")